# 04 - Models: gradient boosting

**Owner:** Punno

This notebook is a question paper. The frame is written. It has **three gaps** to fill in.

| Gap | What goes in it |
|---|---|
| GAP 1 | The model, and the settings to try |
| GAP 2 | Five lines: how the model works |
| GAP 3 | Five lines: what the results show |

**Rules**

1. Fill in the gaps only. Do not change any other cell.
2. Do not load the data again, do not make another split, do not choose the features. The frame does all of this.
3. The notebook must run from the first cell to the last without an error. The last cell prints "All checks passed".
4. Read `docs/dataset_guide.md` first. It says what is in the data and why the frame does what it does.

**What comes out:** the file `results/04_models_punno.csv`, in the same form for every member.

## Frame 1 - Load the clean file and the one split

In [1]:
import os
import json
import numpy as np
import pandas as pd
from sklearn.base import clone, is_classifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score
from sklearn.model_selection import GridSearchCV, LeaveOneGroupOut, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

SEED = 42
CLASSES = ["Will vaccinate", "Only if free", "Undecided or no"]
REPO = "https://raw.githubusercontent.com/SumayerKhan/cse445-hpv-vaccine-ml/main/"


def load(path):
    """Read a file of the project: from the folder on a laptop, from GitHub in Google Colab."""
    local = os.path.join("..", path)
    return pd.read_csv(local if os.path.exists(local) else REPO + path)


df = load("data/hpv_clean.csv")
split = load("data/split.csv")

assert df.shape == (2147, 101), "this is not the clean file"
assert list(split["row"]) == list(range(len(df))), "the split does not match the clean file"

is_train = (split["set"] == "train").to_numpy()
is_test = (split["set"] == "test").to_numpy()
print("train:", is_train.sum(), "parents | test:", is_test.sum(), "parents")

train: 1717 parents | test: 430 parents


## Frame 2 - The features

We run every model on two feature sets.

| Feature set | Features | Why |
|---|---|---|
| main (without barriers) | 92 | The five barrier statements contain the decision itself (dataset guide, 5.5). The main scores are without them. |
| with barriers | 97 | One extra run, to see how much the barrier statements give away |

In [3]:
BARRIERS = ["bar1", "bar2", "bar3", "bar4", "bar5"]
ALL_FEATURES = [c for c in df.columns if not c.startswith(("target_", "meta_"))]

FEATURE_SETS = {
    "main (without barriers)": [c for c in ALL_FEATURES if c not in BARRIERS],
    "with barriers": ALL_FEATURES,
}

y = df["target_3class"]
groups = df["meta_division"]
{name: len(cols) for name, cols in FEATURE_SETS.items()}

{'main (without barriers)': 92, 'with barriers': 97}

## Frame 3 - Preparing the features

A model needs numbers. This part turns the answers into numbers, by the rules of the dataset guide.

| Step | What we do |
|---|---|
| Small categories | Answers that very few parents gave are put together. For example, 26 Christian and 4 Buddhist parents become "Other". |
| Missing values | A number gets the median, a category gets the most common answer |
| Income | Its logarithm, because one income is very large |
| Numbers | Put on the same scale |
| Categories | One-hot encoding: one column for each answer, with 1 or 0 |

All of this is learned from the training rows only.

In [4]:
def merge_small_categories(data):
    """Rule 6 of the dataset guide. A fixed renaming, so nothing is learned from the data here."""
    data = data.copy()
    data["marital_status"] = data["marital_status"].where(data["marital_status"] == "Living with a spouse", "Other")
    data["religion"] = data["religion"].where(data["religion"].isin(["Islam", "Hindu"]), "Other")
    for col in ["occupation", "occup_spouse"]:
        data[col] = data[col].replace("Student", "Others")
    return data


def log_income(values):
    return np.log1p(values)


def make_pipeline(model, columns):
    """Rules 4, 5, 7 and 8 of the dataset guide, then the model. Everything is learned from the training rows only."""
    numbers = [c for c in columns if pd.api.types.is_numeric_dtype(df[c])]
    categories = [c for c in columns if c not in numbers]
    other_numbers = [c for c in numbers if c != "income"]

    prepare = ColumnTransformer([
        ("income", Pipeline([("fill", SimpleImputer(strategy="median")),
                             ("log", FunctionTransformer(log_income)),
                             ("scale", StandardScaler())]), ["income"]),
        ("numbers", Pipeline([("fill", SimpleImputer(strategy="median")),
                              ("scale", StandardScaler())]), other_numbers),
        ("categories", Pipeline([("fill", SimpleImputer(strategy="most_frequent")),
                                 ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), categories),
    ])
    return Pipeline([("prepare", prepare), ("model", model)])


X = merge_small_categories(df[ALL_FEATURES])
print("ready:", X.shape[0], "parents,", X.shape[1], "features")

ready: 2147 parents, 97 features


## GAP 1 - The model and its settings

Fill in the two `...` for each model.

- `"model"`: the scikit-learn classifier, with `random_state=SEED` if it has that setting.
- `"grid"`: the settings to try. Every name starts with `model__`. For example `{"model__max_depth": [3, 5, 10]}` tries three depths. The frame tries every combination on the training rows and keeps the best one.

Try at least two settings, with at least three values each.

- **gradient_boosting:** `GradientBoostingClassifier` or `HistGradientBoostingClassifier` from `sklearn.ensemble`. Settings worth trying: the number of trees, the learning rate, the depth of the trees.

In [6]:
from sklearn.ensemble import GradientBoostingClassifier

MODELS = {
    "gradient_boosting": {
        "model": GradientBoostingClassifier(random_state=SEED),
        "grid": {
            "model__n_estimators": [100, 150, 200],
            "model__learning_rate": [0.03, 0.05, 0.1],
            "model__max_depth": [2, 3, 4],
        },
    },
}

In [7]:
assert isinstance(MODELS, dict) and len(MODELS) > 0, "GAP 1 is empty"
for name, entry in MODELS.items():
    assert entry["model"] is not ... and entry["grid"] is not ..., f"GAP 1 is not filled in for {name}"
    assert is_classifier(entry["model"]), f"{name}: the model must be a scikit-learn classifier"
    assert isinstance(entry["grid"], dict), f"{name}: the grid must be a dictionary"
    assert all(k.startswith("model__") for k in entry["grid"]), f"{name}: every key of the grid must start with model__"
print("GAP 1 is filled in for:", list(MODELS))

GAP 1 is filled in for: ['gradient_boosting']


## Frame 4 - Train and test

For each model and each feature set, the frame does two tests.

| Test | How |
|---|---|
| 80/20 split | Train on the 1,717 training parents, test on the 430 test parents |
| Leave one division out | Train on seven divisions, test on the eighth. Eight times. |

In [8]:
def scores(real, guess):
    recall = recall_score(real, guess, labels=CLASSES, average=None, zero_division=0)
    return {"parents": len(real),
            "accuracy": round(accuracy_score(real, guess) * 100, 1),
            "macro_f1": round(f1_score(real, guess, labels=CLASSES, average="macro", zero_division=0) * 100, 1),
            "right_will_vaccinate": round(recall[0] * 100, 1),
            "right_only_if_free": round(recall[1] * 100, 1),
            "right_undecided_or_no": round(recall[2] * 100, 1),
            "baseline_accuracy": round(real.value_counts(normalize=True).max() * 100, 1)}


results, matrices, chosen = [], {}, {}

for model_name, entry in MODELS.items():
    for set_name, columns in FEATURE_SETS.items():
        assert not [c for c in columns if c.startswith(("target_", "meta_"))], "a target or meta column is in the features"
        pipe = make_pipeline(clone(entry["model"]), columns)

        # Test 1: the 80/20 split. The settings are chosen on the training rows only, with 5-fold cross-validation.
        search = GridSearchCV(pipe, entry["grid"], scoring="f1_macro", n_jobs=-1,
                              cv=StratifiedKFold(5, shuffle=True, random_state=SEED))
        search.fit(X.loc[is_train, columns], y[is_train])
        guess = pd.Series(search.predict(X.loc[is_test, columns]), index=y[is_test].index)
        chosen[(model_name, set_name)] = search.best_params_
        matrices[(model_name, set_name)] = pd.DataFrame(confusion_matrix(y[is_test], guess, labels=CLASSES),
                                                        index=["real: " + c for c in CLASSES],
                                                        columns=["said: " + c for c in CLASSES])
        results.append({"model": model_name, "features": set_name, "test": "80/20 split", "division": "all",
                        **scores(y[is_test], guess), "settings": json.dumps(search.best_params_)})

        # Test 2: leave one division out, with the settings chosen above.
        best = clone(pipe).set_params(**search.best_params_)
        all_guesses = pd.Series(index=y.index, dtype=object)
        for train_rows, test_rows in LeaveOneGroupOut().split(X, y, groups):
            best.fit(X.iloc[train_rows][columns], y.iloc[train_rows])
            all_guesses.iloc[test_rows] = best.predict(X.iloc[test_rows][columns])
            division = groups.iloc[test_rows[0]]
            results.append({"model": model_name, "features": set_name, "test": "leave one division out",
                            "division": division, **scores(y.iloc[test_rows], all_guesses.iloc[test_rows]),
                            "settings": json.dumps(search.best_params_)})
        results.append({"model": model_name, "features": set_name, "test": "leave one division out",
                        "division": "all", **scores(y, all_guesses), "settings": json.dumps(search.best_params_)})
        print("done:", model_name, "|", set_name)

results = pd.DataFrame(results)

done: gradient_boosting | main (without barriers)
done: gradient_boosting | with barriers


## Frame 5 - The results

`baseline_accuracy` is what a model gets that always says the most common class. Read every accuracy next to it.

In [9]:
summary = results[results["division"] == "all"].drop(columns=["division", "settings"])
summary

,model,features,test,parents,accuracy,macro_f1,right_will_vaccinate,right_only_if_free,right_undecided_or_no,baseline_accuracy
0,gradient_boosting,main (without barriers),80/20 split,430,82.1,69.7,94.1,45.1,64.3,70.5
9,gradient_boosting,main (without barriers),leave one division out,2147,66.5,43.5,85.9,22.0,19.0,70.3
10,gradient_boosting,with barriers,80/20 split,430,83.7,73.4,93.7,56.3,64.3,70.5
19,gradient_boosting,with barriers,leave one division out,2147,70.2,48.7,88.7,26.6,26.4,70.3


Which class is mistaken for which, on the 430 test parents:

In [10]:
for (model_name, set_name), matrix in matrices.items():
    print(model_name, "|", set_name, "| 80/20 split | settings:", chosen[(model_name, set_name)])
    display(matrix)

gradient_boosting | main (without barriers) | 80/20 split | settings: {'model__learning_rate': 0.1, 'model__max_depth': 4, 'model__n_estimators': 200}


,said: Will vaccinate,said: Only if free,said: Undecided or no
real: Will vaccinate,285,8,10
real: Only if free,31,32,8
real: Undecided or no,10,10,36


gradient_boosting | with barriers | 80/20 split | settings: {'model__learning_rate': 0.1, 'model__max_depth': 4, 'model__n_estimators': 200}


,said: Will vaccinate,said: Only if free,said: Undecided or no
real: Will vaccinate,284,9,10
real: Only if free,26,40,5
real: Undecided or no,11,9,36


Accuracy when each division is the one left out. In Mymensingh and Rangpur almost every parent is "Will vaccinate", so the baseline there is very high.

In [11]:
by_division = results[(results["test"] == "leave one division out") & (results["division"] != "all")]
table = by_division.pivot_table(index="division", columns=["model", "features"], values="accuracy")
table.columns = [model + " | " + features for model, features in table.columns]
sizes = by_division.drop_duplicates("division").set_index("division")[["parents", "baseline_accuracy"]]
sizes.join(table)

,parents,baseline_accuracy,gradient_boosting | main (without barriers),gradient_boosting | with barriers
division,,,,
Barishal,127,69.3,64.6,66.1
Chattogram,433,48.3,55.4,58.2
Dhaka,560,63.0,67.3,71.1
Khulna,231,79.7,73.6,74.9
Mymensingh,160,98.8,80.6,96.9
Rajshahi,265,62.3,64.5,65.3
Rangpur,228,97.4,76.8,72.4
Sylhet,143,90.9,58.7,75.5


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Frame 6 - Check and save

In [12]:
RESULT_FILE = "04_models_punno.csv"
main = results[results["features"] == "main (without barriers)"]
assert set(results["features"]) == set(FEATURE_SETS) and set(results["model"]) == set(MODELS)
assert results.loc[results["test"] == "80/20 split", "parents"].eq(is_test.sum()).all(), "the 80/20 test must use the 430 test parents"
assert len(results) == len(MODELS) * len(FEATURE_SETS) * (1 + groups.nunique() + 1)

os.makedirs(os.path.join("..", "results"), exist_ok=True)
path = os.path.join("..", "results", RESULT_FILE)
results.to_csv(path, index=False)
print("All checks passed. Saved", path, "|", len(results), "rows")

All checks passed. Saved ../results/04_models_punno.csv | 20 rows


## GAP 2 - How the model works

Five lines, in plain words, not written by an AI tool. Answer these:

1. How does the model decide the class of one parent?
2. What do the settings of GAP 1 change?
3. Which settings did the frame choose, and what does that say?

*Write here.*

## GAP 3 - What the results show

Five lines, with numbers from the tables above. Answer these:

1. Is the model better than the baseline on the 80/20 split? By how much?
2. Which class does it get wrong most often?
3. How much do the scores change with the barrier features?
4. In which divisions is it weak when that division is left out?

*Write here.*